In [8]:
import torch
import torch.nn as nn


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(
        self,
        in_channels,
        intermediate_channels,
        identity_downsample=None,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            intermediate_channels,
            kernel_size=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(intermediate_channels)

        self.conv2 = nn.Conv2d(
            intermediate_channels,
            intermediate_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(intermediate_channels)

        self.conv3 = nn.Conv2d(
            intermediate_channels,
            intermediate_channels * self.expansion,
            kernel_size=1,
            bias=False
        )
        self.bn3 = nn.BatchNorm2d(
            intermediate_channels * self.expansion
        )

        self.relu = nn.ReLU(inplace=True)

        self.identity_downsample = identity_downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)

        out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):
    def __init__(
        self,
        block,
        layers,
        image_channels=3,
        num_classes=1000
    ):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            image_channels,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(64)

        self.relu = nn.ReLU(inplace=True)

        self.maxpool = nn.MaxPool2d(
            kernel_size=3,
            stride=2,
            padding=1
        )

        self.layer1 = self._make_layer(
            block,
            layers[0],
            64,
            stride=1
        )

        self.layer2 = self._make_layer(
            block,
            layers[1],
            128,
            stride=2
        )

        self.layer3 = self._make_layer(
            block,
            layers[2],
            256,
            stride=2
        )

        self.layer4 = self._make_layer(
            block,
            layers[3],
            512,
            stride=2
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(
            512 * block.expansion,
            num_classes
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        x = x.reshape(x.shape[0], -1)

        x = self.fc(x)

        return x

    def _make_layer(
        self,
        block,
        num_residual_blocks,
        intermediate_channels,
        stride
    ):
        identity_downsample = None

        layers = []

        if stride != 1 or self.in_channels != intermediate_channels * 4:
            identity_downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    intermediate_channels * 4,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(
                    intermediate_channels * 4
                )
            )

        layers.append(
            block(
                self.in_channels,
                intermediate_channels,
                identity_downsample,
                stride
            )
        )

        self.in_channels = intermediate_channels * 4

        for _ in range(num_residual_blocks - 1):
            layers.append(
                block(
                    self.in_channels,
                    intermediate_channels
                )
            )

        return nn.Sequential(*layers)


def ResNet50(
    img_channels=3,
    num_classes=1000
):
    return ResNet(
        Bottleneck,
        [3, 4, 6, 3],
        img_channels,
        num_classes
    )


if __name__ == "__main__":
    model = ResNet50()

    x = torch.randn(2, 3, 224, 224)

    y = model(x)

    print(y.shape)

torch.Size([2, 1000])
